# Kaggriculture: What's Actually Worth Doing With a Turn

**The scarce resource is not land or money. It's actions — and there are exactly 720.**

Reading the rules, one asymmetry stands out and seems to decide most of the strategy:

> Each turn, the player may take one action. There are 24 turns per day, and 30 days.
>
> Each turn you can submit up to `maxMarketOrdersPerTurn` (**default 10**) market actions.

A farmer gets **one** action per turn. The market accepts **ten**. So buying and selling is
effectively free, while planting, watering and harvesting are not. Land is capped at 100
tiles and money is recoverable, but a spent turn is gone.

That makes **profit per action** the number that ranks strategies — not profit per tile,
not margin, not payback period.

This notebook computes it for every crop and animal in the spec. Everything below is
derived in-notebook from constants transcribed out of the competition README; where a
rule is ambiguous I say so and show the alternative rather than picking silently.

**Headline:** melons return roughly **9× wheat per action**, and a cared-for sheep is
competitive with melon while needing no replanting. Details, caveats and the arithmetic
follow.

---

## ⚠️ Correction, v2 — and the thing I found while correcting it

**I published v1 of this notebook on 2026-08-09. Sections 1–4 below are unchanged so you
can see exactly what I claimed; sections 5–8 are the correction. The correction turned out
to be less interesting than what fell out of checking it.**

**1. The game economy has been rebalanced twice, and your kernel is probably on neither
version.** Town-centre demand for each product over a season has gone **280 → 140 → 30
units** across `kaggle-environments` 1.29.3 → 1.32.5 → 1.32.6, while the penalty for
gluting the market went *up* fourfold. Section 5 computes all three from transcribed
constants and then **measures which one your notebook is actually running** — and on
Kaggle's stock notebook image, the answer is the oldest of the three. If you are tuning an
agent in a Kaggle notebook without pinning the package, you are tuning against a game that
buys nine times more from you than the current release does.

**2. v1's caveat said the market's price response "is not in the spec — it has to be
measured in-game." That was simply false, and it was the most load-bearing caveat in the
notebook.** The response is fully specified — not in the README, which is what I was
reading, but in the environment source that ships in the `kaggle-environments` package.
`MARKET_PARAMS` gives every product a glut curve and a slope. Section 7 prices the walk-down
exactly, and the answer is not the one I would have guessed.

Section 6 is the finding I did not expect and the reason the ranking above should be read
carefully: **no shop in the game buys melons.**

In [1]:
from dataclasses import dataclass, field
import math
import matplotlib.pyplot as plt
import numpy as np

TURNS_PER_DAY, DAYS = 24, 30
TOTAL_ACTIONS = TURNS_PER_DAY * DAYS
print(f"Season budget: {TURNS_PER_DAY} turns/day x {DAYS} days = {TOTAL_ACTIONS} farmer actions")
print("Market orders are separate (10/turn), so BUY/SELL cost no farmer actions.")

### The correction in one cell, before anything else

Section 5 works through this properly. The short version: `kaggle-environments` has
rebalanced this game twice, and the town's appetite is the thing that moved. Constants
transcribed from each release's `envs/kaggriculture/`; the arithmetic is one loop over the
720 turns of a season.

In [2]:
TURNS = TURNS_PER_DAY * DAYS          # 720 turns in a season

# Transcribed from each release. `schedule` is (day_threshold, multiplier), highest first;
# None means the release has no schedule and the town centre buys a flat 1 per firing.
ECONOMIES = {
    "1.29.3": dict(center_interval=6,  schedule=[(20, 4), (10, 2), (0, 1)], replacement=False),
    "1.32.5": dict(center_interval=12, schedule=[(20, 4), (10, 2), (0, 1)], replacement=False),
    "1.32.6": dict(center_interval=24, schedule=None,                       replacement=True),
}

def town_centre_season_demand(econ):
    """Units of EACH product the town centre removes over one 30-day season."""
    total = 0
    for step in range(TURNS):
        if step % econ["center_interval"] == 0:
            if econ["schedule"] is None:
                total += 1
            else:
                day = step // TURNS_PER_DAY
                total += next(m for thr, m in econ["schedule"] if day >= thr)
    return total

demand = {v: town_centre_season_demand(e) for v, e in ECONOMIES.items()}
for v, d in demand.items():
    print(f"  {v:<8} {d:>4} units of each product bought per season")
print(f"\n  1.29.3 -> 1.32.6 : {demand['1.29.3'] / demand['1.32.6']:.2f}x cut "
      f"({demand['1.29.3'] - demand['1.32.6']} units of demand removed)")

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(list(demand), list(demand.values()), color=["#AAAAAA", "#FF851B", "#FF4136"])
for b, v in zip(bars, demand.values()):
    ax.text(b.get_x() + b.get_width() / 2, v + 6, str(v), ha="center", fontweight="bold")
ax.set_ylabel("units of each product the town centre buys per season")
ax.set_title("The town stopped buying\n"
             "kaggriculture town-centre demand, by kaggle-environments release",
             fontweight="bold")
ax.text(0.5, 0.62, "Kaggle's stock notebook image\nships the one on the left",
        transform=ax.transAxes, ha="center", fontsize=10, color="#0074D9")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 1. The spec, as data

Transcribed from the competition README. Keeping it as a table rather than prose means
every downstream number is recomputed if a constant is wrong — and if you think one *is*
wrong, you can change it here and rerun rather than re-deriving by hand.

In [3]:
@dataclass
class Crop:
    name: str
    seed_cost: int
    price: int          # base market price per unit
    max_yield_day: int  # age at which the yield bonus stops accruing
    max_yield: int      # cap on harvestable units (unfertilized figure in the README)
    ongoing: bool = False
    yield_interval: int = 0   # days between scheduled productions (ongoing only)
    n_yields: int = 0         # number of scheduled productions before decay

# One-time crops. README: "Wheat and Carrot only reach their listed Max Yield of 6 and 4
# with fertilizer; watering alone peaks at 4 and 3." So max_yield here is the watered-only
# figure, which is what the no-fertilizer baseline should use.
CROPS = [
    Crop("Wheat",      10,  25, max_yield_day=4,  max_yield=4),
    Crop("Carrot",     20,  35, max_yield_day=3,  max_yield=3),
    Crop("Melon",      80, 250, max_yield_day=12, max_yield=6),
    Crop("Tomato",     50,  60, max_yield_day=11, max_yield=4, ongoing=True, yield_interval=1, n_yields=4),
    Crop("Strawberry",100, 120, max_yield_day=16, max_yield=4, ongoing=True, yield_interval=2, n_yields=4),
]

@dataclass
class Animal:
    name: str
    cost: int
    price: int          # price per unit of product
    first_yield_day: int
    interval: int       # days between scheduled productions
    max_held: int       # cap on unharvested units on the tile
    build_actions: int = 2   # BUILD_COOP/PASTURE + PLACE

ANIMALS = [
    Animal("Goose/Egg",  300,  50, first_yield_day=4, interval=1, max_held=4),
    Animal("Cow/Milk",   400, 160, first_yield_day=8, interval=2, max_held=6),
    Animal("Sheep/Wool", 500, 200, first_yield_day=6, interval=3, max_held=6),
]

for c in CROPS: print(f"{c.name:<11} seed {c.seed_cost:>4}  price {c.price:>4}")
for a in ANIMALS: print(f"{a.name:<11} cost {a.cost:>4}  price {a.price:>4}  every {a.interval}d")

## 2. Counting the actions a crop really costs

Three action types per one-time crop: one `PLANT`, one `HARVEST`, and however many
`WATER`s. The waterings are where it gets interesting, because two different rules apply:

- **Survival.** A plant that goes two consecutive days unwatered becomes a weed. So you
  must water at least every other day. Note the README's trap: *"A new seed starts with
  `consecutive_unwatered = 1` — the planting day itself counts as the first missed day."*
  You must water on the day you plant, or it dies that night.
- **Yield.** *"Starting at half the plant's `max_yield_day` rounded up, watering during
  the bonus window will add one unit per day."* Only waterings inside that window add yield.

So the cheapest way to a full harvest is: water on planting day, water every *other* day
to stay alive, then water *every* day once the bonus window opens.

In [4]:
def watering_days(crop):
    """Minimum set of days to water for max yield: survive, then saturate the window."""
    window_start = math.ceil(crop.max_yield_day / 2)
    # Yield is base 1 plus one unit per watered day in the window, capped at max_yield.
    days_needed_in_window = min(crop.max_yield - 1, crop.max_yield_day - window_start + 1)
    harvest_day = window_start + days_needed_in_window - 1

    days = {0}                                    # planting day is mandatory
    days |= set(range(window_start, harvest_day + 1))   # daily through the window
    d = 0                                          # every-other-day survival before the window
    while d < window_start:
        days.add(d); d += 2
    return sorted(days), harvest_day

print(f"{'crop':<11} {'window':<9} {'harvest':<8} {'waters':<7} yield")
for c in CROPS:
    if c.ongoing: continue
    w, h = watering_days(c)
    print(f"{c.name:<11} day {math.ceil(c.max_yield_day/2):<5} {h:<8} {len(w):<7} {min(c.max_yield, 1+sum(1 for d in w if d>=math.ceil(c.max_yield_day/2)))}")

Sanity check against the README, which states the answers for two crops: wheat should peak
at **4** units on watering alone and carrot at **3**. If the model above disagrees with
those, the model is wrong — so it is worth checking rather than assuming.

In [5]:
def one_time_yield(crop):
    w, _ = watering_days(crop)
    window_start = math.ceil(crop.max_yield_day / 2)
    bonus = sum(1 for d in w if d >= window_start)
    return min(crop.max_yield, 1 + bonus)

checks = {"Wheat": 4, "Carrot": 3, "Melon": 6}
ok = True
for c in CROPS:
    if c.name in checks:
        got, want = one_time_yield(c), checks[c.name]
        flag = "OK " if got == want else "MISMATCH"
        if got != want: ok = False
        print(f"  {flag} {c.name:<8} model={got}  README={want}")
print("\nmodel reproduces the README's stated yields" if ok else "\nMODEL DISAGREES WITH SPEC — trust the spec, not this notebook")

## 3. Profit per action

Now the actual ranking. For one-time crops: `(units x price - seed) / (plant + waters + harvest)`.

For ongoing crops and animals the shape differs — they produce repeatedly, so the fair
comparison is over their full productive life, including the setup actions.

For animals I model two regimes, because `CARE` changes the picture completely:

- **Feed-only**: `FEED` daily, `HARVEST` on production days. One unit per production.
- **Feed + care**: also `CARE` daily, which banks +1 per day and pays the whole bank out
  on the next production, capped by `max_held`.

In [6]:
def crop_profit_per_action(crop):
    if not crop.ongoing:
        w, harvest_day = watering_days(crop)
        actions = 1 + len(w) + 1                       # PLANT + WATERs + HARVEST
        units = one_time_yield(crop)
        profit = units * crop.price - crop.seed_cost
        return profit / actions, actions, units, harvest_day
    # Ongoing: base 1 per scheduled production, water daily to survive/keep it alive.
    last_day = crop.max_yield_day
    waters = len({0} | set(range(0, last_day + 1, 2)))
    actions = 1 + waters + crop.n_yields               # PLANT + WATERs + one HARVEST each
    units = crop.n_yields
    profit = units * crop.price - crop.seed_cost
    return profit / actions, actions, units, last_day

def animal_profit_per_action(a, care, horizon_days=None):
    """Actions and profit over the animal's productive window within the season."""
    horizon = horizon_days if horizon_days else DAYS
    prod_days = list(range(a.first_yield_day, horizon + 1, a.interval))
    if not prod_days: return 0.0, 0, 0
    feed = horizon - 0                                  # FEED every day it is alive
    care_actions = feed if care else 0
    harvests = len(prod_days)
    actions = a.build_actions + feed + care_actions + harvests

    if not care:
        units = harvests                                # base 1 each
    else:
        # Bank +1 per cared day between productions, paid out at the next production,
        # capped by max_held.
        units, bank = 0, 0
        for d in range(1, horizon + 1):
            bank += 1                                   # fed AND cared this day
            if d in prod_days:
                units += min(1 + bank, a.max_held); bank = 0
    profit = units * a.price - a.cost
    return profit / actions, actions, units

rows = []
for c in CROPS:
    ppa, act, units, _ = crop_profit_per_action(c)
    rows.append((c.name, ppa, act, units, "crop"))
for a in ANIMALS:
    for care in (False, True):
        ppa, act, units = animal_profit_per_action(a, care)
        rows.append((f"{a.name}{' +CARE' if care else ''}", ppa, act, units, "animal"))

rows.sort(key=lambda r: -r[1])
print(f"{'strategy':<20} {'$/action':>9} {'actions':>8} {'units':>6}")
print("-" * 47)
for name, ppa, act, units, _ in rows:
    print(f"{name:<20} {ppa:>9.1f} {act:>8} {units:>6}")

In [7]:
names = [r[0] for r in rows][::-1]
vals  = [r[1] for r in rows][::-1]
cols  = ["#2ECC40" if r[4] == "crop" else "#FF851B" for r in rows][::-1]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(names, vals, color=cols)
ax.set_xlabel("profit per farmer action ($)")
ax.set_title("What a turn is worth, by strategy", fontweight="bold")
for i, v in enumerate(vals):
    ax.text(v + max(vals) * 0.01, i, f"{v:.0f}", va="center", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 4. What this says

Read the chart as a ranking of *what to do with your next turn*, not as a profit forecast.

The gap between the top and bottom is roughly an order of magnitude, which is larger than
most strategic decisions in the game. A player spending turns on wheat while an opponent
spends them on melons is losing ground every single turn, regardless of how well either
plays otherwise.

Three caveats that matter, and none of them are small:

1. **The market is dynamic.** All prices above are the README's *base* prices. Dumping six
   melons at once will move the price against you, and the size of that move is not in the
   spec — it has to be measured in-game. Treat the ranking as an upper bound and a
   priority order, not a revenue estimate.

   > **v2 correction.** The second sentence is wrong. The size of that move *is* specified,
   > in the engine source rather than the README, and section 7 computes it exactly. The
   > first and third sentences stand: base prices are an upper bound and this is a priority
   > order, not a forecast.
2. **Wheat is not competitive as a cash crop, but animals eat it.** Every animal needs
   wheat daily. Wheat's value is as feedstock, and the real comparison is growing it versus
   buying it at market price — a different question from the one this notebook answers.
3. **Time-to-first-yield gates everything early.** Melon takes 10 days to first harvest out
   of a 30-day season, and a sheep takes 6 before it produces anything. With 300–500 up
   front and no income, the opening is a cash-flow problem, not an efficiency problem.

The honest summary: **this tells you what to do with turn 400. It does not tell you what
to do with turn 1.** The opening — how to convert seed money into the first melon or the
first animal without starving — is the harder problem, and it is not solved here.

*Everything above is v1, unchanged. The correction starts here.*

---

# 5. There are three Kaggriculture economies, and your kernel is probably on the oldest

`kaggle-environments` has rebalanced this game twice. The town's appetite has been cut in
two steps — **280 → 140 → 30** units of each product per season, computed in the cell near
the top — and the penalty for gluting the market has gone up fourfold. These are the
constants behind that, transcribed release by release.

| | 1.29.3 | 1.32.5 | 1.32.6 |
|---|---|---|---|
| `townCenterSellInterval` (turns) | 6 | 12 | 24 |
| town-centre demand schedule | `[(20,4),(10,2),(0,1)]` | `[(20,4),(10,2),(0,1)]` | constant removed — flat **1** |
| `townShopSellInterval` (turns) | 2 | 4 | 4 |
| shop unlocks | without replacement | without replacement | **with replacement**, capped at 8 |
| MELON glut `above_target` | 0.90 | 3.60 | 3.60 |

Constants transcribed from each release's `envs/kaggriculture/kaggriculture.py` and
`kaggriculture.json`. Everything downstream is computed from this table.

Now the question that matters more than any of those numbers: **which of these is running
in your notebook?** Don't read the version string — measure it. An episode with two
do-nothing agents produces nothing and sells nothing, so every unit that leaves the market
is town demand, and melon is the cleanest probe available for reasons section 6 explains.

In [8]:
import importlib.metadata as _md
from kaggle_environments import make

try:
    KE_VERSION = _md.version("kaggle-environments")
except Exception:
    KE_VERSION = "unknown"

def idle(obs, cfg):
    return {}

def run_idle(seed=None):
    env = make("kaggriculture", configuration=({"seed": seed} if seed is not None else {}),
               debug=False)
    env.run([idle, idle])
    obs = env.steps[-1][0].observation
    absorbed = {k: 10000 - v for k, v in obs["market"]["inventory"].items()}
    return absorbed, list(obs["town"]["unlocked_shops"])

absorbed, shops = run_idle(seed=1)
melon = absorbed["MELON"]
match = [v for v, d in demand.items() if d == melon]

print(f"kaggle-environments reports version : {KE_VERSION}")
print(f"MELON units absorbed in an idle season: {melon}")
print(f"that demand level belongs to          : {', '.join(match) if match else 'none of the three'}")
print(f"\nshops this season unlocked: {len(set(shops))} distinct out of 8")
print("\nfull idle-season absorption:")
for item, n in absorbed.items():
    print(f"  {item:<11} {n:>5}")

**If that says 1.29.3, and on Kaggle's stock notebook image it does, then every agent you
tune in a Kaggle notebook is being tuned against an economy two rebalances out of date.**
The town buys nine times more from you there than it does under 1.32.6, and punishes a
glut a quarter as hard. A planting schedule that clears comfortably on the stock image can
be sitting on unsellable inventory under the current release.

This is the sort of thing that costs a week and never announces itself: nothing errors, the
numbers just quietly mean something else. `pip install -U kaggle-environments` at the top of
a notebook — with internet enabled — pins it to the current release, and the probe cell
above tells you whether it worked.

I cannot see what version the competition's scorer runs; no notebook can. What the cell
above establishes is narrower and still worth knowing: **the engine in your kernel is not
the newest one, and the three differ enough to change decisions.**

---

# 6. Nobody in town buys melons

The other half of demand is the town's shops, a fixed table in the engine — eight shops,
each with a fixed basket. The table is byte-identical in 1.29.3 and 1.32.6; only the draw
rule changed. Print it and one row is conspicuously empty.

In [9]:
from kaggle_environments.envs.kaggriculture.kaggriculture import SHOPS, PRODUCTS

print("shop baskets, straight out of the engine:")
for name, basket in SHOPS.items():
    print(f"  {name:<15} {', '.join(basket)}")

sellers = {p: [s for s, basket in SHOPS.items() if p in basket]
           for p in PRODUCTS if p != "FERTILIZER"}
print(f"\n{'product':<11} {'shops that buy it':>18}   which")
print("-" * 62)
for p, ss in sorted(sellers.items(), key=lambda kv: len(kv[1])):
    print(f"{p:<11} {len(ss):>18}   {', '.join(ss) if ss else '-- none --'}")

**MELON appears in no shop basket at all.** Its entire demand, for the entire season, is the
town centre — the one buyer whose appetite fell from 280 units to 30 across the two
rebalances. That is also why it is the right probe: no shop noise on top.

This is a direct hit on this notebook's own headline. Section 3 ranks melon first by a wide
margin, and that ranking is arithmetically correct at base price. But it prices six melons
at $250 each into a market that, under 1.32.6, absorbs thirty units in a month — shared
between both players, because there is one market object and both farms sell into it.

The draw rule change makes the rest of the table a lottery. Under 1.32.5 and earlier, eight
unlocks drawn without replacement from eight shops meant **every shop was guaranteed** by
day 24. Under 1.32.6 the same eight draws can hand you the same shop three times, and a
product whose only buyer never comes up has no shop demand for the whole season.

In [10]:
N_SLOTS, N_SHOPS = 8, len(SHOPS)     # 8 unlocks at townShopUnlockInterval=3 within 30 days

print("probability a product gets NO shop buyer all season")
print(f"{'product':<11} {'buyers':>7} {'<= 1.32.5':>11} {'1.32.6':>9}")
print("-" * 41)
for p, ss in sorted(sellers.items(), key=lambda kv: len(kv[1])):
    m = len(ss)
    p_new = ((N_SHOPS - m) / N_SHOPS) ** N_SLOTS    # 8 draws with replacement
    p_old = 1.0 if m == 0 else 0.0                  # without replacement, all 8 unlock
    print(f"{p:<11} {m:>7} {p_old * 100:>10.1f}% {p_new * 100:>8.1f}%")

print(f"\nexpected distinct shops under 1.32.6: "
      f"{N_SHOPS * (1 - ((N_SHOPS - 1) / N_SHOPS) ** N_SLOTS):.2f} of {N_SHOPS} "
      f"(earlier releases always delivered all {N_SHOPS})")

Read the wool row. Wool has exactly one buyer, the yarn store, and under the new draw there
is a **34.4%** chance no yarn store ever opens. Section 3 ranked a cared-for sheep as
competitive with melon; roughly one season in three, under the current release, that sheep
is producing into a market whose only remaining buyer takes one unit a day.

---

# 7. The price response was in the spec all along — just not the spec I read

v1 said the size of the price move "is not in the spec — it has to be measured in-game."
That was wrong, and it is the correction I care most about, because it was never true
rather than made untrue by a patch. The response is a closed form sitting in
`MARKET_PARAMS`, in the same file as everything else:

    price(inv) = base ± amp · f(|inv − I0|),   amp = target · base / f(T)

`f` is one of linear / sq / sqrt / log, chosen per product, with separate curves above and
below the neutral inventory `I0 = 10000`. Selling raises inventory, so the *above* curve is
the one that prices a harvest.

And the market loop in `interpreter()` fills a sell order **one unit at a time**,
re-quoting against inventory after each unit, with each sold unit adding one to inventory.
So a seller walks down their own curve: unit 20 of a batch fetches less than unit 1. That
is not an assumption about how a market ought to work — it is what the loop does, and it
means realised revenue is a sum over the curve rather than `units × base`.

The rebalance quadrupled `above_target` on exactly four products — melon, wool, milk and
strawberry, the four most valuable — and left the other four untouched. Both sets are
transcribed below, so the two regimes can be priced side by side.

In [11]:
import math

# _shape and market_price, transcribed from kaggriculture.py — identical in both releases.
# Note the last line of market_price: max(PRICE_FLOOR, int(round(price))). Prices are
# integers, and rounding is not cosmetic — it moves where the $1 floor is reached on the
# shallow curves. Reproduce it exactly or the numbers below are approximately right, which
# is the wrong kind of right.
PRICE_FLOOR = 1

def _shape(func, x):
    x = max(0.0, x)
    return {"linear": x, "sq": x * x, "sqrt": math.sqrt(x),
            "log": math.log(1.0 + x), "log10": math.log10(1.0 + x)}[func]

def glut_price(base, func, target, T, units_sold):
    amp = target * base / _shape(func, T)
    return max(PRICE_FLOOR, int(round(base - amp * _shape(func, units_sold))))

# above_func / above_target / T, transcribed per release. base and T never changed.
GLUT = {   # product: (base, above_func, T, target_1293, target_1326)
    "MELON":      (250, "sq",     300, 0.90, 3.60),
    "WOOL":       (200, "sq",     105, 0.80, 3.20),
    "MILK":       (160, "linear", 122, 0.40, 1.60),
    "STRAWBERRY": (120, "linear", 100, 0.40, 1.60),
    "TOMATO":     ( 60, "sqrt",   200, 0.60, 0.60),
    "EGG":        ( 50, "log",    332, 0.20, 0.20),
    "CARROT":     ( 35, "sqrt",   450, 0.70, 0.70),
    "WHEAT":      ( 25, "log",    400, 0.20, 0.20),
}

def walk_down(base, func, target, T, k):
    """Revenue from selling k units, quoting each unit the way interpreter() does."""
    revenue, inv = 0.0, 0
    for _ in range(k):
        price = glut_price(base, func, target, T, inv)
        revenue += price
        if price > 1:              # sales at the $1 floor do not add to supply
            inv += 1
    return revenue

K = 20
print(f"fraction of the base-price estimate you actually collect, selling {K} units")
print(f"{'product':<11} {'1.29.3':>8} {'1.32.6':>8}    {'$1 floor at (1.32.6)':>20}")
print("-" * 52)
for p, (base, func, T, t_old, t_new) in GLUT.items():
    r_old = walk_down(base, func, t_old, T, K) / (base * K)
    r_new = walk_down(base, func, t_new, T, K) / (base * K)
    floor = next((k for k in range(1, 3001) if glut_price(base, func, t_new, T, k) <= 1), None)
    print(f"{p:<11} {r_old:>8.2f} {r_new:>8.2f}    {(str(floor) if floor else '>3000'):>20}")

This is the part I got backwards when I guessed before computing. **Melon, the most
expensive product in the game, is the most robust to dumping** — its glut curve is quadratic
against a large `T`, and a quadratic is nearly flat near the origin, so a melon harvest
barely moves its own price. Wool is quadratic against a small `T`, which is the opposite
shape: flat, flat, then a cliff.

So the two corrections point in *different* directions for melon, and that is worth being
precise about rather than tidying away:

- Section 6 says melon's demand is the thinnest in the game — 30 units a season, no shops.
- Section 7 says melon's *price* barely notices a glut.

Both are true and they are about different things. The curve says the market keeps paying
you decently per melon. The demand says the inventory you pushed it to will not come back
down: at one unit a day, a glut you create is still there weeks later, holding the price
below base for every melon you *and your opponent* sell into the same market afterwards.

In [12]:
print("days for the town centre alone to re-absorb a glut you create")
print(f"{'glut':>7} {'1.29.3 (day 20+)':>18} {'1.32.6':>9}")
print("-" * 36)
OLD_LATE = (TURNS_PER_DAY // ECONOMIES["1.29.3"]["center_interval"]) * 4   # firings/day x late multiplier
NEW_RATE = TURNS_PER_DAY // ECONOMIES["1.32.6"]["center_interval"]         # 1/day, flat
for k in (6, 12, 20, 60):
    print(f"{k:>5} u {k / OLD_LATE:>16.1f}d {k / NEW_RATE:>8.0f}d")
print("\n(town centre only; melon has no other buyer, so for melon this is the whole story)")

---

# 8. What I am deliberately not doing

The obvious next step is to redo section 3's ranking with walk-down revenue and 1.32.6
demand, and publish the corrected order. **I am not going to.**

Kaggriculture has a $50,000 prize pool. Documenting what the engine does — the demand
change, the shop table, the price curves, the version your kernel is on — is information
every entrant can read out of a package they have already installed, and getting it wrong
costs people real work for no reason. Handing over a re-optimised strategy ranking is a
different thing, and it is not mine to give away in a competition other people are trying
to win.

So what stands from v1, stated plainly:

- **The thesis holds.** Actions are the scarce resource and profit per action is the right
  frame. Nothing in either rebalance touches that.
- **The section 3 ranking holds as an upper bound at base price.** Section 7 shows the
  bound is tight for melon and loosest for strawberry and milk — and looser under 1.32.6
  than under the engine most people are running.
- **"A cared-for sheep is competitive with melon" is the claim to distrust** — though not
  for the reason I first assumed. Wool is not the worst hit by the walk-down. It is the two
  things stacked behind it: wool is the first product to reach the $1 floor as you sell,
  and it is the product most likely to lose its only buyer to the new shop draw, in
  **34.4%** of seasons.
- **v1's caveat 1 was wrong** and is corrected inline in section 4.

If you spot an error in any of this, say so in the comments and I will fix it and say what
changed — the same way this version does.

---

*Sections 1–4 recompute every figure from constants transcribed out of the competition
README, checked against the two yields the README states outright. Sections 5–8 transcribe
constants from three `kaggle-environments` releases and compute everything from those, so
the numbers render the same wherever this notebook runs; the only cells that depend on the
installed package are the shop table and the probe that identifies which release you have.
If you think a constant or a rule reading is wrong, the tables are the only place to change
it — say so in the comments and I'll fix it.*